In [1]:
import requests
import pandas as pd
import time

In [ ]:
key = ""

In [ ]:
base_url = "https://catalog.api.2gis.com/3.0/items"
headers = {"User-Agent": "Mozilla/5.0"}
location = "37.6173,55.7558"
radius = 100000

results = []

page = 1
while True:
    print(f"\nСтраница {page}")
    params = {
        "q": "пансионат для пожилых",
        "location": location,
        "radius": radius,
        "page": page,
        "page_size": 10,
        "fields": ",".join([
            "items.point",
            "items.address",
            "items.reviews",
            "items.schedule",
            "items.rubrics",
            "items.external_content"
        ]),

        "key": key
    }


    response = requests.get(
        base_url,
        params=params,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()
    data = response.json()

    items = data.get("result", {}).get("items", [])
    if not items:
        break
    for item in items:
        try:
            name = item.get("name")
            address = (item.get("full_address_name") or item.get("address_name"))
            rating = item.get("reviews", {}).get("rating")
            reviews = (item.get("reviews", {}).get("general_review_count"))

            point = item.get("point", {})
            lat = point.get("lat")
            lon = point.get("lon")

            rubrics = []
            for rubric in item.get("rubrics", []):
                rubric_name = rubric.get("name")
                if rubric_name:
                    rubrics.append(rubric_name)
            
            if "Дома престарелых" not in rubrics:
                continue

            results.append({
                "name": name,
                "address": address,
                "rating": rating,
                "reviews": reviews,
                "rubrics": " | ".join(rubrics),
                "lat": lat,
                "lon": lon
            })

        except Exception as e:
            print("Ошибка:", e)

    page += 1
    time.sleep(1)


df = pd.DataFrame(results)
df = df.drop_duplicates(subset=["name", "address"])
df = df.sort_values(by=["reviews", "rating"], ascending=False)

df.to_csv("2gis_pansionaty.csv", index=False, encoding="utf-8-sig")
print(f"\nВсего объектов: {len(df)}")


Страница 1

Страница 2

Страница 3

Страница 4

Страница 5

Страница 6

Всего объектов: 46


In [ ]:
import csv
import time
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options

In [13]:
with open('2gis_pansionaty.csv', 'r', encoding='utf-8-sig') as f:
    orgs = list(csv.DictReader(f))

options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--start-maximized")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

for i, org in enumerate(orgs):
    name = org['name']
    print(f"\n {i+1} {name}")
    
    search_url = f"https://2gis.ru/search/{name.replace(' ', '%20')}"
    driver.get(search_url)
    time.sleep(3)
    
    try:
        link_element = WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='/firm/']")))
        firm_url = link_element.get_attribute('href')
    except Exception as e:
        org['отзывы_собрано'] = 0
        org['текст_отзыва'] = ""
        org['цены'] = ""
        continue
    
    driver.get(firm_url)
    time.sleep(2)
    
    prices = []
    try:
        prices_tab = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, "//span[contains(text(), 'Цены')]")))
        prices_tab.click()
        time.sleep(2)
        price_items = driver.find_elements(By.CSS_SELECTOR, "div[class*='_1lpeuic']")
        
        for item in price_items:
            try:
                service = item.find_element(By.CSS_SELECTOR, "div:first-child").text.strip()
                cost = item.find_element(By.CSS_SELECTOR, "div[class*='_swbsph']").text.strip()
                if service and cost:
                    prices.append(f"{service}: {cost}")
            except:
                continue
        
    except Exception as e:
        print(f"нет вкладки цен: {e}")
    

    try:
        driver.execute_script("window.scrollTo(0, 0)")
        time.sleep(1)
        
        reviews_tab = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, "//span[contains(text(), 'Отзывы')]")))
        reviews_tab.click()
        time.sleep(2)
    except Exception as e:
        org['отзывы_собрано'] = 0
        org['текст_отзыва'] = ""
        org['цены'] = " | ".join(prices) if prices else ""
        continue
    
    rating_current = ''
    reviews_count = ''
    
    try:
        rating_elem = driver.find_element(By.CSS_SELECTOR, "div[class*='_1tam240']")
        rating_current = rating_elem.text.strip()
    except Exception as e:
        None
    
    try:
        reviews_count_elem = driver.find_element(By.CSS_SELECTOR, "div[class*='_1y88ofn']")
        reviews_count_text = reviews_count_elem.text.strip()
        reviews_count = re.search(r'\d+', reviews_count_text).group()
    except Exception as e:
        None

    previous_count = 0
    scroll_attempts = 0
    max_scrolls = 3
    
    while scroll_attempts < max_scrolls:
        current_reviews = driver.find_elements(By.CSS_SELECTOR, "a[class*='_co8kyiw']")
        current_count = len(current_reviews)
        
        if current_count > previous_count:
            previous_count = current_count
        
        try:
            scroll_container = driver.find_element(By.CSS_SELECTOR, "div[data-scroll='true'], div[class*='_8hh56jx']")
            driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", scroll_container)
        except:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
        
        time.sleep(2)
        scroll_attempts += 1
        
        if current_count >= 10:
            break
    

    reviews = []
    review_elements = driver.find_elements(By.CSS_SELECTOR, "a[class*='_co8kyiw']")
    
    for review in review_elements[:15]:
        text = review.text.strip()
        if text and len(text) > 20:
            clean_text = text.replace("Читать целиком", "").replace("Показать полностью", "").strip()
            reviews.append(clean_text[:500])


    
    org['отзывы_собрано'] = len(reviews)
    org['текст_отзыва'] = " | ".join(reviews) if reviews else ""
    org['цены'] = " | ".join(prices) if prices else ""
    org['rating'] = rating_current
    org['reviews'] = reviews_count
    
    time.sleep(2)

driver.quit()


with open('2gis_pansionaty_with_reviews.csv', 'w', newline='', encoding='utf-8-sig') as f:
    fieldnames = ['name', 'address', 'rating', 'reviews', 'rubrics', 'lat', 'lon', 'цены', 'отзывы_собрано', 'текст_отзыва']
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(orgs)




 1 Legacy, пансионат для пожилых людей

 2 Ритм, дом престарелых

 3 Опека Щукинский, пансионат

 4 Теплые беседы, пансионат для пожилых людей

 5 Опека, пансионат Опека Щукинский

 6 Опека Алтуфьевский, пансионат

 7 Идиллия, пансионат для пожилых людей

 8 Опека, Пансионат Опека Красногорский

 9 Академия Долголетия, пансионат для пожилых людей

 10 Академия Долголетия, пансионат для пожилых людей

 11 Бабушки и Дедушки, пансионат для пожилых

 12 Доверие, пансионат для пожилых людей

 13 Академия Долголетия, пансионат для пожилых людей

 14 Тропарево, социальный дом
нет вкладки цен: Message: element click intercepted: Element <span class="_gouffv">...</span> is not clickable at point (398, 445). Other element would receive the click: <svg width="24" height="24" viewBox="0 0 24 24" fill="none" xmlns="http://www.w3.org/2000/svg" size="24" style="transform: rotate(0deg);">...</svg>
  (Session info: chrome=148.0.7778.179); For documentation on this error, please visit: https://www.sele

In [14]:
positive_words = ['хорошо', 'отлично', 'спасибо', 'комфорт', 'чисто', 'внимание', 'забота', 'уют', 'профессионально', 'качественно', 'рекомендую', 'довольны', 'хороший', 'прекрасно', 'вежливые', 'отзывчивые', 'квалифицированные', 'благодарность', 'профессионализм', 'душевно', 'домашняя', 'спокойствие', 'безопасность', 'нравится', 'лучший', 'здорово', 'замечательно', 'великолепно', 'классно', 'супер', 'приятно']
negative_words = ['плохо', 'ужасно', 'грязно', 'грубый', 'невнимание', 'дорого', 'ремонт', 'тесно', 'жалко', 'разочарован', 'неудобно', 'проблем', 'не хватает', 'не нравится', 'ужас', 'кошмар', 'отстой', 'невозможно', 'хамство', 'нерадивые', 'некачественно', 'непрофессионально', 'обман', 'недовольны']

with open('2gis_pansionaty_with_reviews.csv', 'r', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames
    rows = list(reader)

new_fields = ['положительных_слов', 'отрицательных_слов']
for field in new_fields:
    if field not in fieldnames:
        fieldnames.append(field)

for row in rows:
    review_text = row.get('текст_отзыва', '')
    if not review_text or review_text == '':
        row['положительных_слов'] = 0
        row['отрицательных_слов'] = 0
        continue
    
    reviews_list = review_text.split(' | ')
    all_text = review_text.lower()
    words = re.findall(r'\b[а-яА-ЯёЁa-zA-Z]+\b', all_text)

    pos_count = 0
    neg_count = 0
    
    for word in positive_words:
        pos_count += all_text.count(word)
    
    for word in negative_words:
        neg_count += all_text.count(word)
    
    row['положительных_слов'] = pos_count
    row['отрицательных_слов'] = neg_count
    

with open('2gis_pansionaty_with_reviews.csv', 'w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

